In [7]:
import spacy

nlp = spacy.load("en_core_web_md")
with open("data/jci_standards.txt", "r", encoding="utf-8", errors="replace") as f:
    text = f.read()

doc = nlp(text)

In [ ]:
#standardization. Always do so your model is not thrown off.

text = text.replace("`", "'") #the latter is a standard quotation mark.

In [11]:
from spacy.matcher import Matcher
matcher = Matcher(nlp.vocab)
pattern = [
    # Regulatory code like IPSG.2.1 or QPS.10.3
    {"TEXT": {"REGEX": r"^[A-Z]{3}\.\d+(?:\.\d+)*$"}},

    # Optional whitespace between code and clause
    {"IS_SPACE": True, "OP": "*"},

    # Clause text = one or more tokens
    {"OP": "+", "IS_ASCII": True}
]

matcher.add("CLAUSE", [pattern], greedy='LONGEST')
matches = matcher(doc)



In [12]:
print(matches)

[(18231826943396484120, 154, 507), (18231826943396484120, 7583, 7773), (18231826943396484120, 5167, 5328), (18231826943396484120, 5451, 5611), (18231826943396484120, 6263, 6405), (18231826943396484120, 7997, 8128), (18231826943396484120, 4027, 4151), (18231826943396484120, 3802, 3907), (18231826943396484120, 2744, 2846), (18231826943396484120, 4802, 4900), (18231826943396484120, 7867, 7962), (18231826943396484120, 9542, 9637), (18231826943396484120, 1674, 1764), (18231826943396484120, 5807, 5896), (18231826943396484120, 770, 856), (18231826943396484120, 9034, 9119), (18231826943396484120, 5044, 5127), (18231826943396484120, 3104, 3185), (18231826943396484120, 8670, 8749), (18231826943396484120, 3243, 3317), (18231826943396484120, 6900, 6974), (18231826943396484120, 6657, 6728), (18231826943396484120, 9382, 9453), (18231826943396484120, 3955, 4025), (18231826943396484120, 6523, 6592), (18231826943396484120, 61, 129), (18231826943396484120, 1569, 1636), (18231826943396484120, 8868, 8935)

In [13]:
for match_id, start, end in matches:
    span = doc[start:end]
    print(span.text)


APR.4
The hospital permits on-site evaluations of standards and policy compliance or verification of quality and safety
concerns, reports, or regulatory authority sanctions at the discretion of JCI.
Requirement: APR.5
The hospital allows JCI to request (from the hospital or outside agency) and review an original or authenticated
copy of the results and reports of external evaluations from publicly recognized bodies.
Requirement: APR.6
Currently not in effect.
Requirement: APR.7
The hospital selects and uses measures as part of its quality improvement measurement system.
3

--- Page 4 ---
Joint Commission international aCCreditation standards for Hospitals, 6tH edition
Requirement: APR.8
The hospital accurately represents its accreditation status and the programs and services to which JCI
accreditation applies. Only hospitals with current JCI accreditation may display the Gold Seal.
Requirement: APR.9
Any individual hospital staff member (clinical or administrative) can report concerns 

In [15]:
import re
from spacy.tokens import Span

CODE_RE = re.compile(r"^[A-Z]{2,}\.\d+(?:\.\d+)*$")  # allow 2+ letters like APR, GLD, MPE
PAGE_MARKER = re.compile(r"^-{3,}\s*Page", flags=re.IGNORECASE)  # --- Page 4 ---
REQ_TOKEN = "Requirement:"

def extract_clauses(doc, matches):
    clauses = []
    code_token_idxs = {m[1] for m in matches}  # indices of matched code tokens

    for match_id, start, end in matches:
        code_tok = doc[start]
        # clause starts at the token after the code (account for possible intervening whitespace tokens)
        clause_start = start + 1
        if clause_start >= len(doc):
            continue

        # expand until a stop condition occurs
        clause_end = clause_start
        while clause_end < len(doc):
            tok = doc[clause_end]

            # STOP 1: next code-like token (do not include it)
            if CODE_RE.match(tok.text):
                break

            # STOP 2: explicit "Requirement:" token that marks next item
            if tok.text == REQ_TOKEN:
                break

            # STOP 3: page break markers (e.g., '--- Page 4 ---')
            # check both token text and the nearby raw text for visual page separators
            if PAGE_MARKER.match(tok.text) or PAGE_MARKER.match(doc.text[tok.idx: tok.idx + 20]):
                break

            # STOP 4: blank line (two or more newlines) in the whitespace AFTER this token
            if "\n\n" in tok.whitespace_:
                clause_end += 1  # include the token, then stop
                break

            # STOP 5: if token appears to be the start of a new top-level section (heuristic)
            # e.g., uppercase short codes like "GLD.12.1" sometimes appear in the text stream alone
            # (we already cover CODE_RE above but keep this for safety)
            if tok.text.isupper() and len(tok.text) <= 6 and tok.text.endswith("."):
                break

            # otherwise continue
            clause_end += 1

        if clause_end <= clause_start:
            continue

        # Build span and cleaned text
        clause_span = doc[clause_start:clause_end]
        clause_text = clause_span.text.strip()

        # Save result
        clauses.append({
            "code": code_tok.text,
            "clause_text": clause_text,
            "start_token": clause_span.start,
            "end_token": clause_span.end,
            "start_char": clause_span.start_char,
            "end_char": clause_span.end_char,
            "span": clause_span,   # spaCy Span object
        })

    return clauses

matches = matcher(doc)
clauses = extract_clauses(doc, matches)


# Example usage:
# clauses = extract_clauses(doc, matches)
# for c in clauses:
#     print("CODE:", c["code"])
#     print("CLAUSE:", c["clause_text"][:200])  # preview first 200 chars
#     print("---")


In [16]:
for c in clauses:
    print("CODE:", c["code"])
    print("CLAUSE:", c["clause_text"])
    print("----")


CODE: APR.4
CLAUSE: The hospital permits on-site evaluations of standards and policy compliance or verification of quality and safety
concerns, reports, or regulatory authority sanctions at the discretion of JCI.
Requirement:
----
CODE: FMS.9
CLAUSE: The hospital establishes and implements a program to ensure that all utility systems operate
effectively and efficiently.
----
CODE: QPS.2
CLAUSE: Quality and patient safety program staff support the measure selection process throughout the
hospital and provide coordination and integration of measurement activities throughout the
hospital.
----
CODE: PCI.1
CLAUSE: One or more individuals oversee all infection prevention and control activities. This individual(s) is
qualified in infection prevention and control practices through education, training, experience, or
certification.
----
CODE: GLD.3.3
CLAUSE: Hospital leadership ensures that there are uniform programs for the recruitment,
retention, development, and continuing education of all 